# FGSM Evaluation Workflow

This notebook runs the full Google Colab workflow for clean evaluation, FGSM scoring, and clean-vs-adversarial metric comparison with Drive-backed assets.

## 1. Mount Google Drive

Run this first so dataset, checkpoint, and output paths under `/content/drive/MyDrive/...` are available.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('This notebook is intended to run inside Google Colab.') from exc

drive.mount('/content/drive')
Path('/content/drive/MyDrive').exists()

## 2. Enter The Repository And Install Dependencies

In [ ]:
REPO_DIR = Path('/content/cs-199')
if not REPO_DIR.exists():
    raise FileNotFoundError(
        f'Repo directory {REPO_DIR} does not exist. Clone or copy the repository into Colab first.'
    )

%cd /content/cs-199

In [ ]:
!pip install -q -r requirements.txt

## 3. Configure Drive-Backed Paths

Replace `MODEL_WEIGHTS` if your checkpoint filename differs. Leave `TRIAL_FILE = None` to use the default metadata file for the selected split.

In [ ]:
CONFIG_PATH = 'config/WavLM_Nes2Net_ASVspoof5.conf'
DATASET_ROOT = '/content/drive/MyDrive/Education/Subjects/CS 199: Special Problems II/project_storage/data'
METADATA_ROOT = DATASET_ROOT
TRIAL_FILE = None
SSL_PRETRAINED_PATH = '/content/drive/MyDrive/Education/Subjects/CS 199: Special Problems II/project_storage/pretrained_models/WavLM-Large.pt'
MODEL_WEIGHTS = '/content/drive/MyDrive/Education/Subjects/CS 199: Special Problems II/project_storage/checkpoints/model.pth'
OUTPUT_DIR = '/content/drive/MyDrive/Education/Subjects/CS 199: Special Problems II/project_storage/outputs'
MUSAN_PATH = '/content/drive/MyDrive/Education/Subjects/CS 199: Special Problems II/project_storage/musan_data'
RIR_PATH = '/content/drive/MyDrive/Education/Subjects/CS 199: Special Problems II/project_storage/RIR_data'

## 4. Validate The Workflow Paths

In [ ]:
import json
from pathlib import Path

from src.path_utils import resolve_workflow_paths

with open(CONFIG_PATH, 'r') as file:
    config = json.load(file)

config['database_path'] = DATASET_ROOT
config['metadata_path'] = METADATA_ROOT
config['musan_path'] = MUSAN_PATH
config['rir_path'] = RIR_PATH
config['model_config']['ssl_pretrained_path'] = SSL_PRETRAINED_PATH

paths = resolve_workflow_paths(
    config=config,
    output_dir=OUTPUT_DIR,
    model_weights_path=MODEL_WEIGHTS,
    require_training_assets=False,
)

resolved_trial_file = None if TRIAL_FILE is None else Path(TRIAL_FILE).expanduser().resolve(strict=False)
if resolved_trial_file is not None and not resolved_trial_file.is_file():
    raise FileNotFoundError(f'Expected trial file at {resolved_trial_file}, but it was not found.')

paths

## 5. Set Evaluation Parameters

In [ ]:
EVAL_SPLIT = 'eval'  # use 'dev' or 'eval'
EVAL_BATCH_SIZE = 8  # lower this if Colab runs out of memory
FGSM_EPSILON = 0.001
CLAMP_MIN = -1.0
CLAMP_MAX = 1.0
SAVE_ADV_AUDIO = False

## 6. Run Clean Evaluation

In [ ]:
from src.eval_utils import run_clean_evaluation

clean_eval_result = run_clean_evaluation(
    config_path=CONFIG_PATH,
    weights_path=MODEL_WEIGHTS,
    output_dir=OUTPUT_DIR,
    split=EVAL_SPLIT,
    dataset_root=DATASET_ROOT,
    metadata_root=METADATA_ROOT,
    trial_file=TRIAL_FILE,
    ssl_pretrained_path=SSL_PRETRAINED_PATH,
    batch_size=EVAL_BATCH_SIZE,
)
clean_eval_result

## 7. Run FGSM Scoring

In [ ]:
from src.eval_utils import run_fgsm_scoring_pipeline

fgsm_result = run_fgsm_scoring_pipeline(
    config_path=CONFIG_PATH,
    weights_path=MODEL_WEIGHTS,
    output_dir=OUTPUT_DIR,
    split=EVAL_SPLIT,
    dataset_root=DATASET_ROOT,
    metadata_root=METADATA_ROOT,
    trial_file=TRIAL_FILE,
    ssl_pretrained_path=SSL_PRETRAINED_PATH,
    batch_size=EVAL_BATCH_SIZE,
    epsilon=FGSM_EPSILON,
    clamp_min=CLAMP_MIN,
    clamp_max=CLAMP_MAX,
    save_adv_audio=SAVE_ADV_AUDIO,
)
fgsm_result

## 8. Inspect The Comparison Summary

In [ ]:
from pathlib import Path

summary_path = Path(fgsm_result['summary_json_path'])
summary_path.read_text()

In [ ]:
text_summary_path = Path(fgsm_result['summary_text_path'])
print(text_summary_path.read_text())

## 9. Optional CLI Equivalent

Use this if you want to rerun the same flow from a Colab shell cell instead of Python helpers.

In [ ]:
trial_arg = '' if TRIAL_FILE is None else f' --trial-file "{TRIAL_FILE}"'
save_audio_arg = ' --save-adv-audio' if SAVE_ADV_AUDIO else ''
command = f"""python -m attacks.eval_fgsm \
  --config {CONFIG_PATH} \
  --weights \"{MODEL_WEIGHTS}\" \
  --output-dir \"{OUTPUT_DIR}\" \
  --dataset-root \"{DATASET_ROOT}\" \
  --metadata-root \"{METADATA_ROOT}\" \
  --ssl-pretrained-path \"{SSL_PRETRAINED_PATH}\" \
  --split {EVAL_SPLIT} \
  --batch-size {EVAL_BATCH_SIZE} \
  --epsilon {FGSM_EPSILON} \
  --clamp-min {CLAMP_MIN} \
  --clamp-max {CLAMP_MAX}{trial_arg}{save_audio_arg}"""
print(command)